# LIBERO 재학습+재eval — **노드 5 전용** (노드 5대 × GPU 2)

클러스터 = 노드 5대(각 GPU 2). **이 노트북은 노드 5 몫만** 돌린다 (5개 노트북 = 노드 5대).
체크포인트 상태를 보고 **덜 학습(<150k)·미학습은 재학습**, **유효 500ep eval 없는 건 재eval**.

- **유효 eval = overall n_ep ≥ 2500(=500×10)** + action + 체크포인트 ≥150k. 옛 50ep·미완·under-trained eval 은 무효.
- `bimamba_s7`=ours(acm2+carry+BiMamba+overlap), `mosaic`=carry+overlap → config 자동 매핑.
- 순서: **상태 → ① stale 제거 → ② 재학습 → ③ 재eval**. 노드 5대에서 각각 `_node1`~`_node5` 열어 실행.
- ⚠️ ① stale 제거는 공유 FS 면 **한 노드(node1)에서 한 번만**. ⚠️ 500ep eval 은 개당 ~40시간.


In [ ]:
import sys, json, shutil
from pathlib import Path
_here = Path.cwd()
_root = next(c for c in (_here, *_here.parents)
             if (c / 'common_final.py').exists() or (c / 'notebooks' / 'common_final.py').exists())
_root = _root / 'notebooks' if (_root / 'notebooks' / 'common_final.py').exists() else _root
sys.path.insert(0, str(_root))
import importlib, common_final as cf
importlib.reload(cf)

TASK   = 'libero_10'
SEEDS  = [0, 1, 2, 3]
TARGET = cf.CKPT_STEP                     # 150,000 — 이보다 낮으면 재학습
N_TASKS = 10
MIN_VALID_EP = N_TASKS * cf.EVAL_N_EP // 2   # 유효 500ep = overall n_ep >= 2500 (=5000). 옛 50ep(500) 제외

# ── 우리 클러스터: 노드 5대 × GPU 2 (합 10) ──
GPU_COUNTS = [2, 2, 2, 2, 2]
NODE_IDX   = 4               # 이 노트북 = 노드 5 전용 (5개 중 5번째)

# folder_tag → 학습 config 키 (bimamba_s7=ours=acm2+carry+BiMamba+overlap, mosaic=carry+overlap)
MODELS  = ['act', 'acm', 'acm2', 'mosaic', 'bimamba', 'bimamba_s7']
TAG_CFG = {'act': 'act', 'acm': 'acm', 'acm2': 'acm2', 'bimamba': 'bimamba',
           'bimamba_s7': 'bimamba_mosaic', 'mosaic': 'mosaic_infer'}
for folder, cfgkey in TAG_CFG.items():
    cf.v23.MODEL_DIR_NAMES.setdefault(folder, folder)
    if folder not in cf.v23.MODEL_CONFIGS:
        cf.v23.MODEL_CONFIGS[folder] = cf.v23.MODEL_CONFIGS[cfgkey]

TRAIN_ROOT = cf.OUTPUT_BASE / 'train' / TASK
EVAL_ROOT  = cf.OUTPUT_BASE / 'eval_clean' / TASK
print('목표 step:', f'{TARGET:,}', '| 유효 eval: overall n_ep >=', MIN_VALID_EP)
print('정책 매핑:', {k: cf.v23.MODEL_CONFIGS[k][0] for k in ['mosaic', 'bimamba_s7']})

## 상태 분류 — 재학습 / 재eval / stale 제거


In [ ]:
# ── 상태 분류: 재학습 / 재eval / stale제거 대상 ──
def tstep(t, s):
    return cf.v23.last_ckpt_step(TRAIN_ROOT / t / f'seed{s}')

def eval_rec(t, s):
    d = EVAL_ROOT / t / f'seed{s}'
    if not d.is_dir():
        return None
    best = None
    for info in d.rglob('eval_info.json'):
        try:
            ov = json.loads(info.read_text()).get('overall', {})
        except Exception:
            continue
        ne = ov.get('n_ep', ov.get('n_episodes')) or 0
        if best is None or ne > best['n_ep']:
            best = {'n_ep': ne, 'has_act': (info.parent / 'actions').is_dir(), 'info': info}
    return best

train_jobs, eval_pairs, stale_infos, rm_ckpts = [], [], [], []
print(f"{'model':<12}{'seed':>4}{'ckpt':>10}   판정")
print('-' * 52)
for t in MODELS:
    for s in SEEDS:
        step = tstep(t, s)
        needs_train = step is None or step < TARGET
        ev = eval_rec(t, s)
        valid_eval = bool(ev and ev['n_ep'] >= MIN_VALID_EP and ev['has_act'] and not needs_train)
        tags = []
        if needs_train:
            train_jobs.append((t, s, TASK))
            tags.append('재학습' if step else '학습')
            if step is not None:
                rm_ckpts.append(TRAIN_ROOT / t / f'seed{s}')   # under-trained ckpt 제거
        if not valid_eval:
            eval_pairs.append((t, s))
            tags.append('eval')
        if ev and not valid_eval:
            stale_infos.append(ev['info'])                     # 무효 eval_info 제거(재eval 위해)
        if valid_eval:
            tags.append('✅유효')
        print(f'{t:<12}{s:>4}{(f"{step:,}" if step else "-"):>10}   {" + ".join(tags)}')

print('\n' + '=' * 52)
print(f'■ 재학습: {len(train_jobs)}개   ■ 재eval: {len(eval_pairs)}개   ■ stale eval_info 제거: {len(stale_infos)}개')

## ① stale 제거 (under-trained ckpt + 무효 eval_info) — dry-run → EXECUTE=True
**200k done·유효 500ep 은 안 건드림.**


In [ ]:
# ── ① stale 제거: under-trained 체크포인트 + 무효 eval_info ── 확인 후 EXECUTE=True ──
EXECUTE = False

print(f'{"제거" if EXECUTE else "DRY-RUN"} — under-trained ckpt {len(rm_ckpts)} + 무효 eval_info {len(stale_infos)}:')
for d in rm_ckpts:
    print(f'   CKPT {d.relative_to(cf.OUTPUT_BASE)}  (재학습 위해 제거)')
    if EXECUTE and d.is_dir():
        shutil.rmtree(d)
for inf in stale_infos:
    print(f'   INFO {inf.relative_to(EVAL_ROOT)}  (재eval 위해 제거 → skip 안 되게)')
    if EXECUTE and inf.exists():
        inf.unlink()
print('\n' + ('제거 완료.' if EXECUTE else '확인됐으면 EXECUTE=True. (200k done·유효 eval 은 안 건드림)'))

## ② 재학습 (목표 150k, 노드 5×2 GPU)


In [ ]:
# ── ② 재학습 (목표 150k) ── 노드 5대 × GPU 2 로 분할 (상단 NODE_IDX 사용) ──
mine = cf.split_by_gpu(train_jobs, GPU_COUNTS)[NODE_IDX]
gpus = cf.available_gpus()[:GPU_COUNTS[NODE_IDX]]
print(f'노드 {NODE_IDX} | GPU {gpus} | 학습 {len(mine)}개 (전체 {len(train_jobs)}, 분배 {[len(x) for x in cf.split_by_gpu(train_jobs, GPU_COUNTS)]})')
for t, s, _ in mine:
    print(f'   {t:12} seed{s}')
if mine:
    cf.run_training_jobs(mine, gpus=gpus, prefetch_task=TASK)   # 목표 도달분 자동 skip, prefetch 포함

## ③ 재eval (500ep, 노드 5×2 GPU) — ⚠️ 개당 ~40시간
②가 끝난 뒤 실행. 체크포인트 없는 건 자동 skip.


In [ ]:
# ── ③ 재eval (500ep) ── 노드 5대 × GPU 2 분할 ── ⚠️ 개당 ~40시간(5000 에피소드) ──
mine = cf.split_by_gpu(eval_pairs, GPU_COUNTS)[NODE_IDX]
gpus = cf.available_gpus()[:GPU_COUNTS[NODE_IDX]]
print(f'노드 {NODE_IDX} | GPU {gpus} | eval {len(mine)}개 (전체 {len(eval_pairs)}, 분배 {[len(x) for x in cf.split_by_gpu(eval_pairs, GPU_COUNTS)]}) × 500ep')
for t, s in mine:
    print(f'   {t:12} seed{s}')
print('\n⚠️ 500ep = 5000 에피소드 ≈ 개당 40시간. 학습 끝난 뒤, 안 죽는 환경(nohup/충분한 walltime)에서.')
print('   체크포인트 없는 건 자동 skip(학습 먼저), 이미 유효한 건 위 ①에서 안 지웠으니 그대로.')
if mine:
    cf.run_libero_eval_jobs(mine, gpus=gpus, n_episodes=cf.EVAL_N_EP)   # 없는 것만 실행